# New York Crime Data Normalization

This notebook normalizes New York crime data for use with PySpark and Apache Sedona.

**Input**: Raw NY crime CSVs with cryptic column names

**Output**: Clean, normalized parquet files ready for geospatial analysis

## Setup

In [1]:
!pip install pyspark==3.4.2 apache-sedona==1.6.1 -q

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("NYCrimeNormalization")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.apache.sedona:sedona-spark-shaded-3.4_2.12:1.6.1"
    )
    .config(
        "spark.sql.extensions",
        "org.apache.sedona.sql.SedonaSqlExtensions"
    )
    .config(
        "spark.serializer",
        "org.apache.spark.serializer.KryoSerializer"
    )
    .config(
        "spark.kryo.registrator",
        "org.apache.sedona.core.serde.SedonaKryoRegistrator"
    )
    .getOrCreate()
)

## Column Mapping

Original NY crime data has cryptic column names. We map them to clear, descriptive names.

In [3]:
COLUMN_MAPPING = {
    # Identifiers
    "cmplnt_num": "complaint_id",
    "addr_pct_cd": "precinct_code",
    
    # Location
    "boro_nm": "borough",
    "hadevelopt": "housing_development",
    "housing_psa": "housing_psa",
    "parks_nm": "park_name",
    "patrol_boro": "patrol_borough",
    "prem_typ_desc": "premise_type",
    "loc_of_occur_desc": "location_type",
    "station_name": "station_name",
    "transit_district": "transit_district",
    
    # Date/Time
    "cmplnt_fr_dt": "crime_start_date",
    "cmplnt_fr_tm": "crime_start_time",
    "cmplnt_to_dt": "crime_end_date",
    "cmplnt_to_tm": "crime_end_time",
    "rpt_dt": "report_date",
    
    # Crime Classification
    "crm_atpt_cptd_cd": "crime_status",
    "jurisdiction_code": "jurisdiction_code",
    "juris_desc": "jurisdiction_description",
    "ky_cd": "offense_key_code",
    "law_cat_cd": "offense_level",
    "ofns_desc": "offense_description",
    "pd_cd": "internal_code",
    "pd_desc": "internal_description",
    
    # Suspect Info
    "susp_age_group": "suspect_age_group",
    "susp_race": "suspect_race",
    "susp_sex": "suspect_sex",
    
    # Victim Info
    "vic_age_group": "victim_age_group",
    "vic_race": "victim_race",
    "vic_sex": "victim_sex",
    
    # Coordinates
    "x_coord_cd": "x_coordinate",
    "y_coord_cd": "y_coordinate",
    "latitude": "latitude",
    "longitude": "longitude",
    "lat_lon": "lat_lon",
    "geocoded_column": "geometry_wkt"
}

## Load and Normalize Data

In [4]:
import os
from pyspark.sql.functions import col, to_date, to_timestamp, expr, lit, when, year, concat_ws

DATA_PATH = "../data/new_york/crimes/"
OUTPUT_PATH = "../data/new_york/crimes_normalized/"

# List all crime files
crime_files = [f for f in os.listdir(DATA_PATH) if f.endswith('.csv')]
print(f"Found {len(crime_files)} crime files: {crime_files}")

Found 6 crime files: ['new_york_crimes_2022.csv', 'new_york_crimes_2023.csv', 'new_york_crimes_2021.csv', 'new_york_crimes_2020.csv', 'new_york_crimes_2024.csv', 'new_york_crimes_2025.csv']


In [5]:
# Load all files into a single DataFrame
df_raw = spark.read.option("header", True).csv(DATA_PATH + "*.csv")

print(f"Total records: {df_raw.count():,}")
print(f"\nOriginal columns:")
df_raw.printSchema()

Total records: 3,628

Original columns:
root
 |-- cmplnt_num: string (nullable = true)
 |-- addr_pct_cd: string (nullable = true)
 |-- boro_nm: string (nullable = true)
 |-- cmplnt_fr_dt: string (nullable = true)
 |-- cmplnt_fr_tm: string (nullable = true)
 |-- cmplnt_to_dt: string (nullable = true)
 |-- cmplnt_to_tm: string (nullable = true)
 |-- crm_atpt_cptd_cd: string (nullable = true)
 |-- hadevelopt: string (nullable = true)
 |-- housing_psa: string (nullable = true)
 |-- jurisdiction_code: string (nullable = true)
 |-- juris_desc: string (nullable = true)
 |-- ky_cd: string (nullable = true)
 |-- law_cat_cd: string (nullable = true)
 |-- loc_of_occur_desc: string (nullable = true)
 |-- ofns_desc: string (nullable = true)
 |-- parks_nm: string (nullable = true)
 |-- patrol_boro: string (nullable = true)
 |-- pd_cd: string (nullable = true)
 |-- pd_desc: string (nullable = true)
 |-- prem_typ_desc: string (nullable = true)
 |-- rpt_dt: string (nullable = true)
 |-- station_name: s

In [6]:
def normalize_dataframe(df):
    """Rename columns using the mapping and clean data."""
    
    # Rename columns
    for old_name, new_name in COLUMN_MAPPING.items():
        if old_name in df.columns:
            df = df.withColumnRenamed(old_name, new_name)
    
    return df

df_renamed = normalize_dataframe(df_raw)
print("Normalized columns:")
df_renamed.printSchema()

Normalized columns:
root
 |-- complaint_id: string (nullable = true)
 |-- precinct_code: string (nullable = true)
 |-- borough: string (nullable = true)
 |-- crime_start_date: string (nullable = true)
 |-- crime_start_time: string (nullable = true)
 |-- crime_end_date: string (nullable = true)
 |-- crime_end_time: string (nullable = true)
 |-- crime_status: string (nullable = true)
 |-- housing_development: string (nullable = true)
 |-- housing_psa: string (nullable = true)
 |-- jurisdiction_code: string (nullable = true)
 |-- jurisdiction_description: string (nullable = true)
 |-- offense_key_code: string (nullable = true)
 |-- offense_level: string (nullable = true)
 |-- location_type: string (nullable = true)
 |-- offense_description: string (nullable = true)
 |-- park_name: string (nullable = true)
 |-- patrol_borough: string (nullable = true)
 |-- internal_code: string (nullable = true)
 |-- internal_description: string (nullable = true)
 |-- premise_type: string (nullable = true)

## Clean and Transform Data

In [7]:
from pyspark.sql.types import DoubleType, IntegerType, TimestampType

def clean_dataframe(df):
    """Clean and transform data types."""
    
    # Cast coordinate columns to double
    df = df.withColumn("latitude", col("latitude").cast(DoubleType()))
    df = df.withColumn("longitude", col("longitude").cast(DoubleType()))
    df = df.withColumn("x_coordinate", col("x_coordinate").cast(IntegerType()))
    df = df.withColumn("y_coordinate", col("y_coordinate").cast(IntegerType()))
    
    # Cast numeric codes
    df = df.withColumn("precinct_code", col("precinct_code").cast(IntegerType()))
    df = df.withColumn("jurisdiction_code", col("jurisdiction_code").cast(IntegerType()))
    df = df.withColumn("offense_key_code", col("offense_key_code").cast(IntegerType()))
    df = df.withColumn("internal_code", col("internal_code").cast(IntegerType()))
    
    # Parse dates (format: 2020-11-12T00:00:00.000)
    df = df.withColumn("crime_start_date", to_date(col("crime_start_date")))
    df = df.withColumn("crime_end_date", to_date(col("crime_end_date")))
    df = df.withColumn("report_date", to_date(col("report_date")))
    
    # Clean null placeholders
    df = df.replace("(null)", None)
    df = df.replace("UNKNOWN", None)
    
    # Add year column for partitioning
    df = df.withColumn("year", year(col("crime_start_date")))
    
    return df

df_clean = clean_dataframe(df_renamed)
df_clean.printSchema()

root
 |-- complaint_id: string (nullable = true)
 |-- precinct_code: integer (nullable = true)
 |-- borough: string (nullable = true)
 |-- crime_start_date: date (nullable = true)
 |-- crime_start_time: string (nullable = true)
 |-- crime_end_date: date (nullable = true)
 |-- crime_end_time: string (nullable = true)
 |-- crime_status: string (nullable = true)
 |-- housing_development: string (nullable = true)
 |-- housing_psa: string (nullable = true)
 |-- jurisdiction_code: integer (nullable = true)
 |-- jurisdiction_description: string (nullable = true)
 |-- offense_key_code: integer (nullable = true)
 |-- offense_level: string (nullable = true)
 |-- location_type: string (nullable = true)
 |-- offense_description: string (nullable = true)
 |-- park_name: string (nullable = true)
 |-- patrol_borough: string (nullable = true)
 |-- internal_code: integer (nullable = true)
 |-- internal_description: string (nullable = true)
 |-- premise_type: string (nullable = true)
 |-- report_date: d

In [8]:
# Filter rows with valid coordinates for geospatial analysis
df_geo = (
    df_clean
    .filter(col("latitude").isNotNull())
    .filter(col("longitude").isNotNull())
    .filter(col("latitude") != 0)
    .filter(col("longitude") != 0)
)

total = df_clean.count()
valid = df_geo.count()
print(f"Records with valid coordinates: {valid:,} / {total:,} ({100*valid/total:.1f}%)")

Records with valid coordinates: 3,617 / 3,628 (99.7%)


## Add Geometry Column (Apache Sedona)

In [9]:
# Create proper geometry column using Sedona
df_final = df_geo.withColumn(
    "geometry",
    expr("ST_Point(longitude, latitude)")
)

df_final.select("complaint_id", "latitude", "longitude", "geometry").show(5, truncate=False)

+------------+---------+----------+----------------------------+
|complaint_id|latitude |longitude |geometry                    |
+------------+---------+----------+----------------------------+
|299558420   |40.567366|-74.112885|POINT (-74.112885 40.567366)|
|299992069   |40.62547 |-74.151744|POINT (-74.151744 40.62547) |
|302130633   |40.560927|-74.195362|POINT (-74.195362 40.560927)|
|298728053   |40.612956|-74.124704|POINT (-74.124704 40.612956)|
|301409060   |40.553778|-74.183771|POINT (-74.183771 40.553778)|
+------------+---------+----------+----------------------------+
only showing top 5 rows



## Preview Normalized Data

In [10]:
# Show sample of key columns
df_final.select(
    "complaint_id",
    "borough",
    "crime_start_date",
    "offense_level",
    "offense_description",
    "premise_type",
    "latitude",
    "longitude",
    "year"
).show(10, truncate=30)

+------------+-------------+----------------+-------------+-----------------------------+-------------------+-----------------+------------------+----+
|complaint_id|      borough|crime_start_date|offense_level|          offense_description|       premise_type|         latitude|         longitude|year|
+------------+-------------+----------------+-------------+-----------------------------+-------------------+-----------------+------------------+----+
|   299558420|STATEN ISLAND|      2024-12-13|       FELONY|                GRAND LARCENY|               BANK|        40.567366|        -74.112885|2024|
|   299992069|STATEN ISLAND|      2024-11-30|  MISDEMEANOR|UNAUTHORIZED USE OF A VEHICLE|        CHAIN STORE|         40.62547|        -74.151744|2024|
|   302130633|STATEN ISLAND|      2024-11-15|    VIOLATION|                HARRASSMENT 2|     BAR/NIGHT CLUB|        40.560927|        -74.195362|2024|
|   298728053|STATEN ISLAND|      2024-12-31|    VIOLATION|                HARRASSMENT 2

In [11]:
# Summary statistics
print("Records by Borough:")
df_final.groupBy("borough").count().orderBy(col("count").desc()).show()

print("\nRecords by Offense Level:")
df_final.groupBy("offense_level").count().orderBy(col("count").desc()).show()

print("\nRecords by Year:")
df_final.groupBy("year").count().orderBy("year").show()

Records by Borough:
+-------------+-----+
|      borough|count|
+-------------+-----+
|STATEN ISLAND| 1501|
|     BROOKLYN|  720|
|    MANHATTAN|  680|
|       QUEENS|  389|
|        BRONX|  311|
|         NULL|   16|
+-------------+-----+


Records by Offense Level:
+-------------+-----+
|offense_level|count|
+-------------+-----+
|       FELONY| 1570|
|  MISDEMEANOR| 1543|
|    VIOLATION|  504|
+-------------+-----+


Records by Year:
+----+-----+
|year|count|
+----+-----+
|2020|  158|
|2021|  201|
|2022|  399|
|2023|  864|
|2024| 1000|
|2025|  995|
+----+-----+



## Save Normalized Data

In [12]:
# Drop redundant columns and save as parquet (partitioned by year)
columns_to_keep = [
    "complaint_id",
    "precinct_code",
    "borough",
    "crime_start_date",
    "crime_start_time",
    "crime_end_date",
    "crime_end_time",
    "report_date",
    "crime_status",
    "jurisdiction_description",
    "offense_level",
    "offense_description",
    "internal_description",
    "premise_type",
    "location_type",
    "patrol_borough",
    "suspect_age_group",
    "suspect_race",
    "suspect_sex",
    "victim_age_group",
    "victim_race",
    "victim_sex",
    "latitude",
    "longitude",
    "year"
]

df_export = df_final.select(columns_to_keep)

# Save as parquet partitioned by year
df_export.write.mode("overwrite").partitionBy("year").parquet(OUTPUT_PATH + "parquet")
print(f"Saved parquet files to: {OUTPUT_PATH}parquet")

Saved parquet files to: ../data/new_york/crimes_normalized/parquet


In [13]:
# Also save as single CSV for easy inspection
df_export.coalesce(1).write.mode("overwrite").option("header", True).csv(OUTPUT_PATH + "csv")
print(f"Saved CSV to: {OUTPUT_PATH}csv")

Saved CSV to: ../data/new_york/crimes_normalized/csv


In [14]:
# Verify the output
df_verify = spark.read.parquet(OUTPUT_PATH + "parquet")
print(f"Verified {df_verify.count():,} records in parquet output")
df_verify.printSchema()

Verified 3,617 records in parquet output
root
 |-- complaint_id: string (nullable = true)
 |-- precinct_code: integer (nullable = true)
 |-- borough: string (nullable = true)
 |-- crime_start_date: date (nullable = true)
 |-- crime_start_time: string (nullable = true)
 |-- crime_end_date: date (nullable = true)
 |-- crime_end_time: string (nullable = true)
 |-- report_date: date (nullable = true)
 |-- crime_status: string (nullable = true)
 |-- jurisdiction_description: string (nullable = true)
 |-- offense_level: string (nullable = true)
 |-- offense_description: string (nullable = true)
 |-- internal_description: string (nullable = true)
 |-- premise_type: string (nullable = true)
 |-- location_type: string (nullable = true)
 |-- patrol_borough: string (nullable = true)
 |-- suspect_age_group: string (nullable = true)
 |-- suspect_race: string (nullable = true)
 |-- suspect_sex: string (nullable = true)
 |-- victim_age_group: string (nullable = true)
 |-- victim_race: string (nullabl

## Usage Example with Apache Sedona

After running this normalization, you can use the clean data like this:

In [15]:
# Example: Load normalized data and create geometry for Sedona
df_crimes = spark.read.parquet(OUTPUT_PATH + "parquet")

# Add geometry column for Sedona operations
df_crimes_geo = df_crimes.withColumn(
    "geometry",
    expr("ST_Point(longitude, latitude)")
)

# Register as temp view for SQL queries
df_crimes_geo.createOrReplaceTempView("ny_crimes")

# Example spatial query
spark.sql("""
    SELECT borough, offense_level, COUNT(*) as total
    FROM ny_crimes
    WHERE year = 2023
    GROUP BY borough, offense_level
    ORDER BY total DESC
""").show()

+-------------+-------------+-----+
|      borough|offense_level|total|
+-------------+-------------+-----+
|    MANHATTAN|       FELONY|  111|
|     BROOKLYN|       FELONY|  107|
|       QUEENS|       FELONY|  107|
|     BROOKLYN|  MISDEMEANOR|   98|
|        BRONX|       FELONY|   82|
|    MANHATTAN|  MISDEMEANOR|   74|
|       QUEENS|  MISDEMEANOR|   68|
|        BRONX|  MISDEMEANOR|   66|
|     BROOKLYN|    VIOLATION|   33|
|       QUEENS|    VIOLATION|   29|
|    MANHATTAN|    VIOLATION|   24|
|STATEN ISLAND|       FELONY|   21|
|STATEN ISLAND|  MISDEMEANOR|   21|
|        BRONX|    VIOLATION|   15|
|STATEN ISLAND|    VIOLATION|    6|
|         NULL|  MISDEMEANOR|    2|
+-------------+-------------+-----+



In [16]:
spark.stop()

---

## 📦 Save to DVC (Data Version Control)

After running this notebook, track the normalized data with DVC for version control and cloud storage.

### Step 1: Track with DVC
From the **project root directory**, run:
```bash
cd /Users/felipebriosso/Documents/ORT/Tesis-Repo

# Track the parquet files (recommended - smaller, faster)
dvc add notebooks/data/new_york/crimes_normalized/parquet

# Or track CSV if you prefer
dvc add notebooks/data/new_york/crimes_normalized/csv
```

This creates `.dvc` files that act as pointers to your data.

### Step 2: Push to Cloud Storage
```bash
dvc push
```

This uploads the data to Google Drive (configured remote).

### Step 3: Commit to Git
```bash
git add notebooks/data/new_york/crimes_normalized/*.dvc
git add notebooks/data/new_york/crimes_normalized/.gitignore
git commit -m \"Add normalized NY crime data (2020-2025)\"
git push
```

### For Collaborators to Download
```bash
git pull           # Get the .dvc pointer files
dvc pull           # Download the actual data from Google Drive
```

---

## 📁 Output Structure

The normalized data is saved as:

```
notebooks/data/new_york/crimes_normalized/
├── parquet/                    # Partitioned by year (RECOMMENDED)
│   ├── year=2020/
│   ├── year=2021/
│   ├── year=2022/
│   ├── year=2023/
│   ├── year=2024/
│   └── year=2025/
└── csv/                        # Single CSV for inspection
    └── part-00000-*.csv
```

**Use parquet/** for:
- PySpark analysis (much faster)
- Apache Sedona geospatial operations
- Large-scale data processing

**Use csv/** for:
- Quick data inspection
- Sharing with non-technical stakeholders
- Importing to Excel/other tools"